# PaMCP — Colab experiment runner (E1 first)

This notebook executes **E1: Registry characterization** on the frozen merged MCP registry. It clones the exact research branch, records the Git commit and input hashes, creates a frozen input snapshot, and exports a run bundle to Google Drive.

> **Scope control:** this notebook runs E1 only. E2–E22 remain `planned_not_run` until their independent labels, probes, runtimes, or evaluation oracles are available and validated. Do not use the legacy E0–E7 replay pipeline for definitive results.

E1 is metadata-only: no benchmark gold tools, silver effect labels, LLM records, policy outcomes, or MCP server calls are read or inferred.

## 1. Configure and clone the pinned experiment source


In [ ]:
from pathlib import Path
import shutil, subprocess, sys, json, os

REPOSITORY = 'https://github.com/msellamiTN/taas-provenance-discovery.git'
BRANCH = 'research/pamcp-experimental-v1'
REPO_DIR = Path('/content/taas-provenance-discovery')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPOSITORY, str(REPO_DIR)], check=True)
actual_branch = subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=REPO_DIR, text=True).strip()
commit_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert actual_branch == BRANCH, f'Expected {BRANCH}, got {actual_branch}'
assert (REPO_DIR / 'data/processed/registry_merged.json').is_file(), 'Frozen merged registry is missing'
assert (REPO_DIR / 'docs/experiment_design/pamcp_experiments_v1.json').is_file(), 'Experiment catalogue is missing'
print({'branch': actual_branch, 'commit': commit_sha})

## 2. Install dependencies and validate the experiment catalogue
The install uses the repository's declared `experiment` extra. No LLM API key or live MCP credentials are required for E1.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[experiment]'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, 'experiments/catalog_cli.py', 'validate'], cwd=REPO_DIR, check=True)
print('Environment and E1–E22 catalogue validation passed.')

## 3. Mount Google Drive for durable run artifacts
Authorize the Drive mount when prompted. Outputs are stored outside the cloned repository under `MyDrive/PaMCP_Experiments/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_ROOT = Path('/content/drive/MyDrive/PaMCP_Experiments')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Durable output directory:', OUTPUT_ROOT)

## 4. Execute E1 — Registry characterization
The runner computes source/category distributions, server/tool counts, metadata completeness, annotation-hint distributions, JSON Schema type distributions, name collisions, and published definition-hash integrity.

In [ ]:
sys.path.insert(0, str(REPO_DIR / 'experiments' / 'colab'))
from e1_registry_characterization import run_e1

RUN_DIR = run_e1(REPO_DIR, OUTPUT_ROOT)
summary = json.loads((RUN_DIR / 'summary.json').read_text(encoding='utf-8'))
manifest = json.loads((RUN_DIR / 'MANIFEST.json').read_text(encoding='utf-8'))
print('E1 run directory:', RUN_DIR)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Registry SHA-256:', manifest['inputs']['registry_sha256'])
print('Git commit:', manifest['execution']['git_commit'])

## 5. Inspect the outputs
These are descriptive registry properties, not safety, effectiveness, or runtime claims.

In [ ]:
import pandas as pd
from IPython.display import display, Image
for file_name in ['source_distribution.csv', 'category_by_source.csv', 'annotation_hint_distribution.csv', 'schema_type_frequency.csv']:
    print('\n' + file_name)
    display(pd.read_csv(RUN_DIR / file_name))
plot_path = RUN_DIR / 'category_by_source.png'
if plot_path.exists():
    display(Image(filename=str(plot_path)))

## 6. Package and hash the run bundle
The ZIP contains the frozen compressed registry input, source pins, result tables, plot, summary, and manifest.

In [ ]:
from IPython.display import FileLink, display
import hashlib
archive_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR))
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print({'artifact_zip': str(archive_path), 'sha256': archive_sha256, 'bytes': archive_path.stat().st_size})
display(FileLink(str(archive_path)))
print('E1 completed. E2–E22 have not been run or marked complete.')

## 7. Scientific interpretation boundary

Report only descriptive properties supported by these tables. A mismatch between a published hash and a recomputed hash is a data-integrity finding to investigate, not proof of malicious behavior. MCP annotations remain untrusted metadata.

**Next experiment gate:** E2 requires an independently human-annotated semantic-typing subset and frozen adjudication protocol. E3 requires credential-free runnable reference servers and an isolated probe harness. E4–E7 require independent safety/evidence references and policy-matched evaluation.